# Smoothing Techniques in NLP

Smoothing techniques are used in Natural Language Processing to handle the problem of zero probability estimates for unseen n-grams and improve model generalization.

## Zero Probability Problem in N-grams

When training language models on limited data, we encounter the **zero probability problem**: n-grams that don't appear in the training data receive zero probability, which causes issues in probability calculations.

### Example: Unsmoothed Bigram Model

In [ ]:
from collections import defaultdict

# Training data
training_data = ["the cat sat", "the dog ran", "a cat meowed"]

# Count bigrams
bigram_counts = defaultdict(int)
for sentence in training_data:
    words = sentence.split()
    for i in range(len(words) - 1):
        bigram = (words[i], words[i+1])
        bigram_counts[bigram] += 1

# Calculate probabilities (unsmoothed)
total_bigrams = sum(bigram_counts.values())
bigram_probs = {bigram: count / total_bigrams for bigram, count in bigram_counts.items()}

print("Bigram Probabilities:")
for bigram, prob in bigram_probs.items():
    print(f"{bigram}: {prob:.3f}")

# Query unseen bigram
unseen_bigram = ("the", "bird")
prob_unseen = bigram_probs.get(unseen_bigram, 0)
print(f"\nProbability of unseen bigram {unseen_bigram}: {prob_unseen}")


Bigram Probabilities:
('the', 'cat'): 0.167
('cat', 'sat'): 0.167
('the', 'dog'): 0.167
('dog', 'ran'): 0.167
('a', 'cat'): 0.167
('cat', 'meowed'): 0.167

Probability of unseen bigram ('the', 'bird'): 0
This zero probability causes problems in language modeling!



This demonstrates how unsmoothed models assign zero probability to unseen n-grams, breaking downstream applications like speech recognition or machine translation.

In [5]:
import nltk
from collections import defaultdict
import pandas as pd

In [6]:
import kagglehub
import os

# This downloads the dataset to your local cache
dataset_dir = kagglehub.dataset_download("ltcmdrdata/plain-text-wikipedia-202011")

# List all files inside the dataset
print("Files in dataset:")
for root, dirs, files in os.walk(dataset_dir):
    for file in files:
        # Print the relative paths you can use for dataset_load
        print(os.path.relpath(os.path.join(root, file), dataset_dir))

/media/rabi/9662E4B562E49B6F/projects/NLP_Techniques/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Files in dataset:
enwiki20201020/d9771113-ea97-46de-a812-27a5605ab884.json
enwiki20201020/6512b959-2b7b-42f7-a4e2-32734b33cd13.json
enwiki20201020/b2faad9e-dfa8-4107-bc35-0694725c10a5.json
enwiki20201020/f1dc01bf-7678-4453-b0ea-ad9ce9d5f084.json
enwiki20201020/301cdca4-a982-414d-b26d-1483028d7861.json
enwiki20201020/dd53bacd-ce88-4f03-9b06-ed75c9dec21e.json
enwiki20201020/98ba4472-9b7b-4168-b9e2-2b88393163da.json
enwiki20201020/14f66e46-f1fc-4d3d-8890-07d04d00e9dd.json
enwiki20201020/577897f1-71bf-45d2-a656-571504d8d40e.json
enwiki20201020/918bf1d8-97ca-4b91-9b64-b072775f960c.json
enwiki20201020/4e0d9991-5307-4c9b-bf98-979152fb67af.json
enwiki20201020/0b5e8bc0-af6c-48a3-8138-84557d78284b.json
enwiki20201020/b9826a38-4511-4ca7-b776-9e99e3cd9da1.json
enwiki20201020/2c3b989a-e17c-4041-97fe-ae3f3e1c4c9f.json
enwiki20201020/0d782c05-94c3-4454-9a6d-9978af11c419.json
enwiki20201020/998d802a-4092-45cc-ae9d-82691c9c7354.json
enwiki20201020/2ea5a70e-be96-485f-9c50-a6051cca4dde.json
enwiki2020102

In [7]:
import json
import os
from collections import defaultdict, Counter

full_file_path = os.path.join(dataset_dir, "enwiki20201020/0aaf9aae-5557-466c-8c50-13170a3cbec8.json")

# Counter is faster and more idiomatic than defaultdict(int) for unigrams
unigram_counts = Counter()
bigram_counts = defaultdict(Counter)
vocab = set()

with open(full_file_path, 'r', encoding='utf-8') as f:

    data = json.load(f)

    for item in data:
        # .get() safely handles missing keys without throwing errors
        text = item.get('text')

        if not text or not isinstance(text, str):
            continue

        tokens = text.lower().split()
        if not tokens:
            continue

        vocab.update(tokens)

        # Pad tokens once
        padded_tokens = ['<s>'] + tokens + ['</s>']

        # Fast unigram counting (runs in C-level)
        unigram_counts.update(padded_tokens)

        # Fast bigram counting using zip()
        # This avoids the slow `tokens[i]` index lookups in pure Python
        for w1, w2 in zip(padded_tokens, padded_tokens[1:]):
            bigram_counts[w1][w2] += 1


In [8]:
word1 = "hawthorne"
word2 = "academy"
word3 = "road"

# Access the count for the bigram (word1, word2)
# The bigram_counts is structured as bigram_counts[word1][word2]
bigram_count_unseen = bigram_counts[word1][word2]
bigram_count_seen = bigram_counts[word1][word3]

print(f"Count of bigram ('{word1}', '{word2}'): {bigram_count_unseen}")
print(f"Count of bigram ('{word1}', '{word3}'): {bigram_count_seen}")


Count of bigram ('hawthorne', 'academy'): 0
Count of bigram ('hawthorne', 'road'): 3


# Smoothing Techniques in NLP

When training language models on limited data, we encounter the **zero probability problem**. N-grams that do not appear in the training data receive a probability of zero, which severely affects probability calculations and breaks downstream applications like speech recognition or machine translation. Smoothing techniques are used to handle these zero probability estimates for unseen n-grams and improve overall model generalization.



Here are the smoothing techniques explored in the code:

### 1. Laplace Smoothing (Add-One Smoothing)
* **Description:** This is a simple baseline smoothing method that adds 1 to all n-gram counts before normalization. For bigrams, the probability $P(w_2|w_1)$ is calculated as $\frac{C(w_1, w_2) + 1}{C(w_1) + V}$, where $V$ is the vocabulary size.
* **Use Cases:** It is typically used for language modeling with small datasets and in Naive Bayes text classification.


In [9]:
class LaplaceBigramModel:
    def __init__(self, bigram_counts, unigram_counts, vocab):
        self.bigram_counts = bigram_counts
        self.unigram_counts = unigram_counts
        self.V = len(vocab) + 2  # include <s> and </s>

    def get_prob(self, w1, w2):
        """Returns the Laplace smoothed probability P(w2 | w1)"""
        denom = self.unigram_counts.get(w1, 0) + self.V

        if w1 in self.bigram_counts:
            num = self.bigram_counts[w1].get(w2, 0) + 1
        else:
            num = 1

        return num / denom

# --- Usage ---
model = LaplaceBigramModel(bigram_counts, unigram_counts, vocab)
prob_unseen = model.get_prob("hawthorne", "academy")
prob_seen = model.get_prob("hawthorne", "road")
print(f"Laplace smoothed probability P('academy' | 'hawthorne'): {prob_unseen:.10f}")
print(f"Laplace smoothed probability P('road' | 'hawthorne'): {prob_seen:.7f}")

Laplace smoothed probability P('academy' | 'hawthorne'): 0.0000051335
Laplace smoothed probability P('road' | 'hawthorne'): 0.0000205



### 2. Add-k Smoothing
* **Description:** A generalized, more flexible version of Laplace smoothing that adds a small constant $k$ (where $0 < k < 1$) instead of 1. The adjusted probability is calculated as $\frac{C(w_1, w_2) + k}{C(w_1) + k \times V}$.
* **Use Cases:** Because $k$ is tunable for specific datasets, it helps reduce the severe over-smoothing effects caused by standard Add-One smoothing.


In [10]:
class AddkBigramModel:
    def __init__(self, bigram_counts, unigram_counts, vocab, k=0.5):
        self.bigram_counts = bigram_counts
        self.unigram_counts = unigram_counts
        self.V = len(vocab) + 2  # Include <s> and </s>
        self.k = k

    def get_prob(self, w1, w2):
        """Returns the Add-k smoothed probability P(w2 | w1)"""
        # Numerator: count(w1, w2) + k
        # Denominator: count(w1) + k * V

        denom = self.unigram_counts.get(w1, 0) + self.k * self.V

        if w1 in self.bigram_counts:
            num = self.bigram_counts[w1].get(w2, 0) + self.k
        else:
            num = self.k # if w1 is unseen, bigram_counts[w1] would not exist

        # Handle the case where the denominator could be zero if w1 is completely unseen and k*V is 0 (though k*V should generally not be 0)
        if denom == 0:
            return 0.0

        return num / denom


In [11]:
# --- Usage ---
# Instantiate the Add-k model with k=0.5
addk_model = AddkBigramModel(bigram_counts, unigram_counts, vocab, k=0.5)

# Get the probability of an unseen bigram
prob_addk = addk_model.get_prob("hawthorne", "academy")
print(f"Add-k Probability for ('hawthorne', 'academy') with k=0.5: {prob_addk:.10f}")

# Get the probability of a seen bigram
prob_addk_seen = addk_model.get_prob("hawthorne", "road")
print(f"Add-k Probability for ('hawthorne', 'road') with k=0.5: {prob_addk_seen:.7f}")

Add-k Probability for ('hawthorne', 'academy') with k=0.5: 0.0000051334
Add-k Probability for ('hawthorne', 'road') with k=0.5: 0.0000359


### 3. Good-Turing Smoothing
* **Description:** Good-Turing smoothing adjusts the observed counts of n-grams by examining the frequency distribution of frequencies. It reassigns the probability mass from n-grams that appear $r$ times to n-grams that appear $r-1$ times, with special handling for unseen n-grams (r=0). The adjusted count is calculated as 
$r^* = (r+1) \frac{N_{r+1}}{N_r}$,
where $N_r$ is the number of n-grams appearing exactly $r$ times.
* **Use Cases:** Particularly effective for small datasets and when the frequency distribution is sparse. It's widely used in machine translation and information retrieval.

In [20]:
from collections import defaultdict

class GoodTuringBigramModel:
    def __init__(self, bigram_counts, unigram_counts, vocab):
        self.bigram_counts = bigram_counts
        self.unigram_counts = unigram_counts
        self.V = len(vocab) + 2  # Include <s> and </s>
        
        # Precompute frequency of frequencies for bigrams
        self.bigram_freq_of_freqs = self._compute_freq_of_freqs(bigram_counts)
        self.adjusted_bigram_counts = self._compute_adjusted_counts(bigram_counts, self.bigram_freq_of_freqs)
        
        # --- NEW LOGIC FOR UNSEEN BIGRAMS ---
        # N1: Number of bigrams seen exactly once
        self.N1 = self.bigram_freq_of_freqs.get(1, 0)
        
        # N0: Number of unseen bigrams (Total possible bigrams - Total seen bigram types)
        total_possible_bigrams = self.V * self.V
        seen_bigram_types = sum(self.bigram_freq_of_freqs.values())
        self.N0 = total_possible_bigrams - seen_bigram_types
        
        # Adjusted count for r=0: (1 * N1) / N0
        self.adjusted_zero_count = self.N1 / self.N0 if self.N0 > 0 else 0
        
    def _compute_freq_of_freqs(self, bigram_counts):
        """Compute N_r: the number of bigrams appearing exactly r times"""
        freq_of_freqs = defaultdict(int)
        for w1 in bigram_counts:
            for w2, count in bigram_counts[w1].items():
                freq_of_freqs[count] += 1
        return freq_of_freqs
    
    def _compute_adjusted_counts(self, bigram_counts, freq_of_freqs):
        """Compute adjusted counts r* = (r+1) * N_{r+1} / N_r"""
        adjusted_counts = {}
        for w1 in bigram_counts:
            adjusted_counts[w1] = {}
            for w2, count in bigram_counts[w1].items():
                if count in freq_of_freqs and (count + 1) in freq_of_freqs:
                    adjusted_count = (count + 1) * freq_of_freqs[count + 1] / freq_of_freqs[count]
                else:
                    adjusted_count = count  # Fall back to original count if N_{r+1} unavailable
                adjusted_counts[w1][w2] = adjusted_count
        return adjusted_counts
    
    def get_prob(self, w1, w2):
        """Returns the Good-Turing smoothed probability P(w2 | w1)"""
        
        # --- NEW FALLBACK LOGIC ---
        # Get adjusted count if seen, otherwise use the adjusted count for unseen items
        if w1 in self.adjusted_bigram_counts and w2 in self.adjusted_bigram_counts[w1]:
            adjusted_count = self.adjusted_bigram_counts[w1][w2]
        else:
            adjusted_count = self.adjusted_zero_count 
        
        unigram_count = self.unigram_counts.get(w1, 0)
        
        if unigram_count == 0:
            return 1.0 / self.V  # Uniform probability for unseen context
        
        return adjusted_count / unigram_count
# --- Usage ---
good_turing_model = GoodTuringBigramModel(bigram_counts, unigram_counts,
                                             vocab)
prob_good_turing_unseen = good_turing_model.get_prob("hawthorne", "academy")
prob_good_turing_seen = good_turing_model.get_prob("hawthorne", "road")
print(f"Good-Turing Probability for unseen bigram ('hawthorne', 'academy'): {prob_good_turing_unseen:.10f}")
print(f"Good-Turing Probability for seen bigram ('hawthorne', 'road'): {prob_good_turing_seen:.7f}")


Good-Turing Probability for unseen bigram ('hawthorne', 'academy'): 0.0000044854
Good-Turing Probability for seen bigram ('hawthorne', 'road'): 0.5108935



### 4. Simple Backoff Smoothing
* **Description:** If an n-gram is unseen, backoff smoothing "backs off" to a lower-order n-gram model to estimate the probability. For example, if a bigram count is zero, the model relies on the unigram probability instead.


In [13]:
class AddkUnigramModel:
    def __init__(self, unigram_counts, vocab, k=0.5):
        self.unigram_counts = unigram_counts
        self.V_unigram = len(vocab) + 2  # Include <s> and </s> for vocabulary size
        self.total_words = sum(unigram_counts.values())  # Total count of all unigrams
        self.k = k

    def get_prob(self, word):
        """Returns the Add-k smoothed probability P(word)"""
        num = self.unigram_counts.get(word, 0) + self.k
        denom = self.total_words + self.k * self.V_unigram
        if denom == 0:
            return 0.0
        return num / denom

class SimpleBackoffBigramModel:
    def __init__(self, bigram_counts, unigram_counts, vocab, k_bigram=0.5, k_unigram=0.5):
        self.bigram_counts = bigram_counts
        self.addk_bigram_model = AddkBigramModel(bigram_counts, unigram_counts, vocab, k=k_bigram)
        self.addk_unigram_model = AddkUnigramModel(unigram_counts, vocab, k=k_unigram)

    def get_prob(self, w1, w2):
        """Returns the backoff smoothed probability P(w2 | w1)"""
        # Check the actual count of the bigram (w1, w2)
        actual_bigram_count = self.bigram_counts.get(w1, {}).get(w2, 0)

        if actual_bigram_count > 0:
            # If the bigram has been seen, use the Add-k smoothed bigram probability
            return self.addk_bigram_model.get_prob(w1, w2)
        else:
            # If the bigram has not been seen (count is 0), back off to the
            # Add-k smoothed unigram probability of w2.
            # (Note: A more rigorous backoff would involve a scaling factor
            # to ensure probabilities sum to 1, but this demonstrates the core concept.)
            return self.addk_unigram_model.get_prob(w2)

In [14]:

# --- Usage ---
# Instantiate the Backoff model
backoff_model = SimpleBackoffBigramModel(bigram_counts, unigram_counts, vocab, k_bigram=0.5, k_unigram=0.5)

# Get the probability for an unseen bigram (should backoff to unigram)
prob_backoff_unseen = backoff_model.get_prob("hawthorne", "academy")
print(f"Backoff Probability for ('hawthorne', 'academy'): {prob_backoff_unseen:.7f}")

# Get the probability for a seen bigram (should use smoothed bigram prob)
prob_backoff_seen = backoff_model.get_prob("hawthorne", "road")
print(f"Backoff Probability for ('hawthorne', 'road'): {prob_backoff_seen:.7f}")

# Let's also get the unigram probability for 'academy' to compare
unigram_model = AddkUnigramModel(unigram_counts, vocab, k=0.5)
prob_academy_unigram = unigram_model.get_prob("academy")
print(f"Add-k Unigram Probability for ('academy'): {prob_academy_unigram:.7f}")

Backoff Probability for ('hawthorne', 'academy'): 0.0001226
Backoff Probability for ('hawthorne', 'road'): 0.0000359
Add-k Unigram Probability for ('academy'): 0.0001226



### 5. Interpolation Smoothing
* **Description:** Instead of choosing between different n-gram orders, interpolation continuously mixes the probability estimates from all available models (e.g., combining unigram and bigram probabilities) using weighted coefficients (like $\lambda_1$).


In [15]:
class InterpolationBigramModel:
    def __init__(self, bigram_model, unigram_model, lambda_1=0.7):
        self.bigram_model = bigram_model
        self.unigram_model = unigram_model
        self.lambda_1 = lambda_1
        self.lambda_2 = 1 - lambda_1 # Assuming simple linear interpolation between two models

    def get_prob(self, w1, w2):
        """Returns the interpolated probability P(w2 | w1)"""
        # P_interp(w2 | w1) = lambda_1 * P_bigram(w2 | w1) + lambda_2 * P_unigram(w2)

        # Get smoothed bigram probability from the underlying bigram model
        prob_bigram = self.bigram_model.get_prob(w1, w2)

        # Get smoothed unigram probability from the underlying unigram model
        prob_unigram = self.unigram_model.get_prob(w2)

        # Combine them using the lambda weights
        interpolated_prob = (self.lambda_1 * prob_bigram) + (self.lambda_2 * prob_unigram)
        return interpolated_prob



In [16]:

# --- Usage ---
# First, ensure you have your AddkBigramModel and AddkUnigramModel instances
# We'll use the ones already created or create new ones for clarity

# Addk Bigram Model (can be Laplace or any other smoothed bigram model)
addk_bigram_model_for_interpolation = AddkBigramModel(bigram_counts, unigram_counts, vocab, k=0.5)

# Addk Unigram Model
addk_unigram_model_for_interpolation = AddkUnigramModel(unigram_counts, vocab, k=0.5)

# Instantiate the Interpolation model with a lambda value (e.g., 0.7 for bigram, 0.3 for unigram)
interpolation_model = InterpolationBigramModel(
    bigram_model=addk_bigram_model_for_interpolation,
    unigram_model=addk_unigram_model_for_interpolation,
    lambda_1=0.7
)

# Get the interpolated probability for an unseen bigram
prob_interp_unseen = interpolation_model.get_prob("hawthorne", "academy")
print(f"Interpolated Probability for ('hawthorne', 'academy'): {prob_interp_unseen:.7f}")

# Get the interpolated probability for a seen bigram
prob_interp_seen = interpolation_model.get_prob("hawthorne", "road")
print(f"Interpolated Probability for ('hawthorne', 'road'): {prob_interp_seen:.7f}")


Interpolated Probability for ('hawthorne', 'academy'): 0.0000404
Interpolated Probability for ('hawthorne', 'road'): 0.0001082



### 6. Kneser-Ney Smoothing
* **Description:** Considered one of the most effective smoothing methods, Kneser-Ney discounts a small absolute value (e.g., $d=0.75$) from the raw counts of seen n-grams. It then redistributes this probability mass to lower-order models based on "continuation probabilities"—evaluating how likely a word is to appear in novel contexts rather than just its raw frequency.


In [17]:
class KneserNeyBigramModel:
    def __init__(self, bigram_counts, unigram_counts, vocab, d=0.75): # d is the discount factor
        self.bigram_counts = bigram_counts
        self.unigram_counts = unigram_counts
        self.vocab = vocab
        self.V = len(vocab) + 2 # Including <s> and </s>
        self.d = d # Discount factor

        # Precompute the number of unique types that follow each word (for the P_continuation component)
        self.unique_followers = defaultdict(int)
        for w1 in bigram_counts:
            self.unique_followers[w1] = len(bigram_counts[w1])

        # Precompute the total count of word types that a word 'w' is a follower of
        self.follower_types_count = defaultdict(int)
        for w1 in bigram_counts:
            for w2 in bigram_counts[w1]:
                self.follower_types_count[w2] += 1

        # Total number of unique bigrams
        self.N1_plus_bigrams = sum(len(counts) for counts in bigram_counts.values())

    def get_prob(self, w1, w2):
        """Returns the Kneser-Ney smoothed probability P_KN(w2 | w1)"""
        count_w1w2 = self.bigram_counts.get(w1, {}).get(w2, 0)
        count_w1 = self.unigram_counts.get(w1, 0)

        # Calculate the lower-order probability P_continuation(w2)
        # This is a key part of Kneser-Ney: how many different previous words does w2 have?
        numerator_pc = self.follower_types_count.get(w2, 0)
        denominator_pc = self.N1_plus_bigrams if self.N1_plus_bigrams > 0 else 1 # Avoid division by zero
        p_continuation_w2 = numerator_pc / denominator_pc

        # Calculate the normalization factor alpha(w1)
        # alpha(w1) = (d / count(w1)) * |{w': count(w1, w') > 0}| (number of unique words following w1)
        if count_w1 == 0:
            alpha_w1 = 0 # If w1 hasn't been seen, alpha should be 0, and we only use P_continuation
        else:
            alpha_w1 = (self.d / count_w1) * self.unique_followers.get(w1, 0)

        if count_w1w2 > 0: # Bigram (w1, w2) has been seen
            # Discounted bigram probability
            prob = (count_w1w2 - self.d) / count_w1 + alpha_w1 * p_continuation_w2
        else: # Bigram (w1, w2) has not been seen (or count_w1w2 == 0 after discounting)
            prob = alpha_w1 * p_continuation_w2

        return max(0.0, prob) # Ensure probability is non-negative


# --- Usage ---
# Instantiate the Kneser-Ney model
kneser_ney_model = KneserNeyBigramModel(bigram_counts, unigram_counts, vocab, d=0.75)

# Get the Kneser-Ney probability for an unseen bigram
prob_kn_unseen = kneser_ney_model.get_prob("hawthorne", "academy")
print(f"Kneser-Ney Probability for ('hawthorne', 'academy'): {prob_kn_unseen:.7f}")

# Get the Kneser-Ney probability for a seen bigram
prob_kn_seen = kneser_ney_model.get_prob("hawthorne", "road")
print(f"Kneser-Ney Probability for ('hawthorne', 'road'): {prob_kn_seen:.7f}")


Kneser-Ney Probability for ('hawthorne', 'academy'): 0.0000445
Kneser-Ney Probability for ('hawthorne', 'road'): 0.5626282


# Comparison of Smoothing Techniques — Results & Summary

Below is a compact comparison of the probabilities for the two example bigrams: **("hawthorne", "academy")** *(unseen)* and **("hawthorne", "road")** *(seen)*.

| Model | $P(\text{'academy'} \mid \text{'hawthorne'})$ | $P(\text{'road'} \mid \text{'hawthorne'})$ | Notes |
| --- | --- | --- | --- |
| **Laplace (Add-1)** | 0.0000051335 | 0.0000205 | Simple smoothing; guarantees non-zero probability for unseen events but often over-smooths. |
| **Add-k (k=0.5)** | 0.0000051334 | 0.0000359 | Tunable smoothing using parameter $k$; less aggressive than Laplace when $k < 1$. |
| **Good-Turing** | 0.0000044854 | 0.5108935 | Redistributes probability mass using frequency-of-frequencies; effective for sparse data. |
| **Simple Backoff** | 0.0001226 | 0.0000359 | Falls back to unigram probabilities when the bigram is unseen. |
| **Interpolation** | 0.0000404 | 0.0001082 | Weighted combination of bigram and unigram probabilities. |
| **Kneser-Ney** | 0.0000445 | 0.5626282 | Advanced smoothing method using discounting and continuation probabilities. |

---

## Summary

* The **unsmoothed bigram model** assigns probability **0** to unseen bigrams, while all smoothing methods assign a **non-zero** probability.
* **Laplace smoothing** generally assigns the **largest probability mass** to unseen events. If `prob_unseen` is significantly larger than the unseen probabilities from other methods, this indicates **over-smoothing**.
* **Add-k smoothing** reduces over-smoothing compared to Laplace. Smaller values of $k$ produce smaller probabilities for unseen events.
* **Good-Turing smoothing** estimates unseen probabilities using the ratio of singleton counts and total unseen events:

$$\frac{N_1}{N_0}$$

This usually results in a more balanced and principled unseen probability.

* **Simple Backoff** approximates the unseen bigram probability using the unigram probability:

$$P_{\text{backoff}}(\text{academy} \mid \text{hawthorne}) \approx P(\text{academy})$$

* **Interpolation smoothing** combines unigram and bigram information:

$$P_{\text{interp}}(w_i \mid w_{i-1}) = \lambda P_{\text{bigram}} + (1-\lambda)P_{\text{unigram}}$$

Depending on the value of $\lambda$, the result may lean more toward the unigram or bigram estimate.

* **Kneser-Ney smoothing** is often considered one of the strongest traditional smoothing techniques for language modeling because it incorporates continuation probabilities and discounted counts.

---

For **small experiments** or **baseline models**, we can use:
* **Add-k smoothing**
* **Good-Turing smoothing**


For **language modeling tasks** where predictive accuracy is important, we can use:
* **Kneser-Ney smoothing**
* **Interpolation models** incorporating Kneser-Ney probabilities

After comparing the numerical values carefully, we find that very high unseen probabilities result in **over-smoothing**, and extremely low probabilities for seen events are a sign of **under-smoothing**.
